# 📚 RAG Book Q&A — Computer Science: An Overview (13th ed.)

> **Sistema de preguntas y respuestas (RAG) sobre un libro técnico en inglés, traducido al español.**

Este notebook implementa un pipeline completo de **Retrieval Augmented Generation (RAG)**:

```
PDF (inglés)
   └──► [1] Extracción de texto
            └──► [2] Chunking (~1500 fragmentos, overlap)
                     └──► [3] Traducción EN→ES con deep-translator (Google)
                              └──► [4] Embeddings con Gemini (gemini-embedding-001, 768 dim)
                                       └──► [5] Almacenamiento en ChromaDB persistente
                                                └──► [6] Pregunta del usuario (ipywidgets)
                                                         └──► [7] Embedding de la pregunta
                                                                  └──► [8] Búsqueda semántica (top-k)
                                                                           └──► [9] LLM (gemini-2.5-flash) + contexto → Respuesta
```

**Autor:** *Patrick Jusephy Melo Ramos*  
**Stack:** `pypdf` · `langchain` · `tiktoken` · `deep-translator` · `google-genai` · `chromadb` · `ipywidgets`

---

## 🧱 Paso 0 — Instalación de dependencias

Si ya instalaste con `pip install -r requirements.txt` puedes saltar esta celda.

| Librería | Para qué |
|---|---|
| `pypdf` | Leer y extraer texto del PDF |
| `tiktoken` | Contar tokens (limita el tamaño de cada chunk) |
| `langchain-text-splitters` | Dividir el texto en fragmentos con `chunk_overlap` |
| `deep-translator` | Traducir EN→ES (gratis, sin Docker, sin API key) |
| `google-genai` | SDK oficial de Gemini (embeddings + chat) |
| `chromadb` | Base de datos vectorial persistente |
| `ipywidgets` | Casilla interactiva para la pregunta del usuario |
| `tqdm` | Barra de progreso al traducir e indexar |
| `python-dotenv` | Cargar la API key desde un archivo `.env` (opcional) |

In [1]:
# Descomenta esta línea solo si NO instalaste con `pip install -r requirements.txt`
# %pip install -r requirements.txt

## 🔐 Paso 1 — Configuración de la API Key de Gemini

Necesitas una API key gratuita de **Google AI Studio**: <https://aistudio.google.com/app/apikey>

> ⚠️ El plan gratuito de `gemini-embedding-001` permite **~60 requests por minuto**. Por eso el código incluye un `time.sleep(1.1)` entre embeddings para no excederlo.

**Tres formas de proporcionar la key (en orden de prioridad):**

1. **Variable de entorno** `GEMINI_API_KEY` (recomendado para producción)
2. **Archivo `.env`** en la raíz del proyecto con la línea: `GEMINI_API_KEY=tu_clave_aqui`
3. **Pegarla manualmente** cuando el notebook te la pida (modo interactivo seguro con `getpass`)

👉 **AQUÍ ES DONDE COLOCAS TU API KEY** si decides usar el método 3.

In [2]:
import os
from getpass import getpass

# Intentamos cargar desde .env si existe
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    # 👇 PEGA TU API KEY AQUÍ cuando el notebook te lo pida (no se mostrará en pantalla)
    GEMINI_API_KEY = getpass("🔑 Ingresa tu GEMINI_API_KEY: ")

# Guardamos como variable de entorno para que google-genai la detecte
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

assert GEMINI_API_KEY, "❌ No se ha definido GEMINI_API_KEY"
print("✅ API Key cargada correctamente (longitud:", len(GEMINI_API_KEY), "caracteres)")

✅ API Key cargada correctamente (longitud: 39 caracteres)


## 🌐 Paso 2 — Probar el traductor (deep-translator)

Usamos `deep-translator` con `GoogleTranslator`: usa el motor web de Google Translate **gratis**, sin API key, sin Docker, sin instalación adicional. Solo necesita conexión a internet.

**Características importantes:**
- ✅ **Sin rate limit estricto** — pero hacemos `time.sleep(0.3)` entre llamadas para evitar baneo de IP de Google
- ✅ **Límite de 5000 caracteres por request** — nuestros chunks tienen ~1000 caracteres, todo en orden
- ✅ **Soporta `translate_batch()`** — útil si quieres acelerar más adelante

Probamos que funciona traduciendo una frase corta:

In [3]:
from deep_translator import GoogleTranslator

def translator_health_check():
    """Verifica que deep-translator funciona."""
    try:
        result = GoogleTranslator(source="en", target="es").translate("Hello world")
        print(f"✅ deep-translator funciona correctamente.")
        print(f"   Prueba: 'Hello world' → '{result}'")
        return True
    except Exception as e:
        print(f"❌ Error en deep-translator: {e}")
        print(f"   Verifica tu conexión a internet.")
        return False

translator_health_check()

✅ deep-translator funciona correctamente.
   Prueba: 'Hello world' → 'Hola Mundo'


True

## 📖 Paso 3 — Extracción de texto del PDF

Usamos `pypdf` para extraer el texto plano del libro. **Las imágenes y fórmulas se ignoran automáticamente**: `extract_text()` solo devuelve la capa de texto del PDF, no procesa elementos gráficos ni símbolos matemáticos en formato vectorial.

> 📌 Si el PDF tiene fórmulas o iconos, simplemente no aparecerán en el texto extraído. No tenemos que hacer nada especial para ignorarlos: la librería ya lo hace por nosotros.

In [4]:
from pypdf import PdfReader

# 👇 Ruta al PDF — cámbiala si tu libro está en otra ubicación
PDF_PATH = r"C:\Users\LENOVO\Downloads\computer-science-an-overview-13th-ed.pdf"


def pdf_to_string(path: str) -> str:
    """
    Carga el PDF y concatena el texto extraíble de cada página.
    Ignora imágenes, iconos y fórmulas (pypdf solo devuelve texto plano).
    """
    document_text = ""
    loaded_pdf = PdfReader(path)
    print(f"📄 Páginas en el PDF: {len(loaded_pdf.pages)}")
    for page in loaded_pdf.pages:
        document_text += page.extract_text() or ""
    return document_text


book_text = pdf_to_string(PDF_PATH)
print(f"📊 Caracteres extraídos: {len(book_text):,}")
print(f"📊 Palabras aprox.: {len(book_text.split()):,}")
print(f"\n🔍 Primeros 500 caracteres:\n{book_text[:500]}")

📄 Páginas en el PDF: 737
📊 Caracteres extraídos: 1,790,480
📊 Palabras aprox.: 288,011

🔍 Primeros 500 caracteres:
Computer Science
An Overview
THIRTEENTH EDITION
 J. Glenn Brookshear 
Dennis Brylow
GLOBAL 
EDITION
GLOBAL 
EDITION
Computer Science 
An Overview
Brookshear  
Brylow 
THIRTEENTH 
EDITION
GLOBAL 
EDITION
This is a special edition of an established title widely used by colleges and 
universities throughout the world. Pearson published this exclusive edition 
for the benefit of students outside the United States and Canada. If you 
purchased this book within the United States or Canada, you should 


## 🔢 Paso 4 — Tokenización

Los LLMs no leen palabras, leen **tokens** (secuencias numéricas que representan grupos de caracteres). Necesitamos contar tokens para que cada chunk no exceda el límite del modelo.

Usamos el tokenizer `cl100k_base` de OpenAI como referencia (es muy similar al de Gemini y nos sirve para estimar tamaños).

In [5]:
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base")

def token_counter(text: str) -> int:
    """Devuelve el número de tokens en un texto."""
    return len(tokenizer.encode(text))

total_tokens = token_counter(book_text)
print(f"🔢 Total de tokens en el libro: {total_tokens:,}")
print(f"   (gemini-embedding-001 acepta hasta 8 192 tokens por request,")
print(f"    por eso necesitamos hacer chunking)")

🔢 Total de tokens en el libro: 415,629
   (gemini-embedding-001 acepta hasta 8 192 tokens por request,
    por eso necesitamos hacer chunking)


## ✂️ Paso 5 — Chunking con `chunk_overlap`

Dividimos el libro en fragmentos pequeños usando `RecursiveCharacterTextSplitter`. Este splitter intenta cortar primero en saltos de párrafo (`\n\n`), luego en puntos (`.`), después en saltos de línea (`\n`), y finalmente en espacios — para evitar romper palabras u oraciones a la mitad.

**Parámetros clave:**

- `chunk_size=256` → cada fragmento tendrá ~256 tokens (≈ 1 párrafo). Texto pequeño = búsqueda semántica más precisa.
- `chunk_overlap=30` → cada fragmento **comparte 30 tokens con el siguiente**. Esto evita perder contexto en los bordes (si una idea queda partida entre dos chunks, ambos tienen el contexto necesario).
- Objetivo: **~1500 chunks**, lo suficiente para encajar en el límite gratuito de Gemini (60 req/min × 25 min ≈ 1500).

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=30,
    length_function=token_counter,
    separators=["\n\n", ".", "\n", " ", ""],
)

chunks = text_splitter.create_documents(
    [book_text],
    metadatas=[{
        "book": "Computer Science: An Overview",
        "edition": "13th",
        "author": "J. Glenn Brookshear, Dennis Brylow",
        "language": "en->es",
    }],
)

print(f"✂️  Total de chunks generados: {len(chunks)}")
print(f"\n🔍 Ejemplo — chunk #50 (en inglés, antes de traducir):\n")
print(chunks[50].page_content)

✂️  Total de chunks generados: 1574

🔍 Ejemplo — chunk #50 (en inglés, antes de traducir):

.
In the next two chapters, we look at ways data can be organized within a 
computer system. In Chapter 8 (Data Abstractions), we introduce techniques 
traditionally used for organizing data in a computer’s main memory and 
then trace the evolution of data abstraction from the concept of primitives 
M00_BROO3427_13_GE_C00.indd   26 17/10/18   4:37 PM 270.4 The Overarching Themes of Computer Science
to today’s object-oriented techniques. In Chapter 9 (Database Systems), we 
consider methods traditionally used for organizing data in a computer’s mass 
storage and investigate how extremely large and complex database systems 
are implemented.
In Chapter 10 (Computer Graphics), we explore the subject of graphics and 
animation, a field that deals with creating and photographing virtual worlds. 
Based on advancements in the more traditional areas of computer science 
such as machine architecture, algo

## 🌍 Paso 6 — Traducir cada chunk EN → ES con deep-translator

Iteramos sobre los chunks y traducimos uno por uno. Como `GoogleTranslator` usa el motor web de Google sin clave, hacemos un `time.sleep(0.3)` entre llamadas para no parecer un bot agresivo y evitar baneo temporal de IP.

> 💡 Tiempo estimado: ~10-15 minutos para 1500 chunks.

**Estrategia anti-fallos:**
- `try/except` con 3 reintentos por chunk
- Si todos los reintentos fallan, dejamos el texto original en inglés (mejor algo que nada)
- Guardamos el resultado a disco (`pickle`) cada 50 chunks para no perder progreso

In [8]:
import time
import pickle
import os
from tqdm.auto import tqdm
from deep_translator import GoogleTranslator

TRANSLATED_CACHE = "translated_chunks.pkl"
TRANSLATE_SLEEP = 0.3  # segundos entre llamadas para no saturar Google


def translate_text(text: str, source: str = "en", target: str = "es") -> str:
    """Traduce un texto usando deep-translator. Si falla 3 veces, devuelve el original."""
    for attempt in range(3):
        try:
            return GoogleTranslator(source=source, target=target).translate(text) or text
        except Exception as e:
            if attempt < 2:
                time.sleep(2 * (attempt + 1))  # backoff exponencial
            else:
                print(f"⚠️  Falló traducción tras 3 intentos: {str(e)[:80]}")
    return text  # fallback


def save_progress(data, path):
    with open(path, "wb") as f:
        pickle.dump(data, f)


# Cargar cache si existe
if os.path.exists(TRANSLATED_CACHE):
    with open(TRANSLATED_CACHE, "rb") as f:
        translated_chunks = pickle.load(f)
    print(f"♻️  Cargados {len(translated_chunks)} chunks traducidos desde cache.")

    # Si el cache está incompleto, continuamos desde donde quedamos
    if len(translated_chunks) < len(chunks):
        print(f"   Faltan {len(chunks) - len(translated_chunks)} chunks por traducir. Continuando...")
        for chunk in tqdm(chunks[len(translated_chunks):], desc="🌍 Traduciendo (continuación)"):
            translated_text = translate_text(chunk.page_content)
            translated_chunks.append({
                "text": translated_text,
                "original": chunk.page_content,
                "metadata": chunk.metadata,
            })
            time.sleep(TRANSLATE_SLEEP)
            if len(translated_chunks) % 50 == 0:
                save_progress(translated_chunks, TRANSLATED_CACHE)
        save_progress(translated_chunks, TRANSLATED_CACHE)
else:
    translated_chunks = []
    for i, chunk in enumerate(tqdm(chunks, desc="🌍 Traduciendo EN→ES")):
        translated_text = translate_text(chunk.page_content)
        translated_chunks.append({
            "text": translated_text,
            "original": chunk.page_content,
            "metadata": chunk.metadata,
        })
        time.sleep(TRANSLATE_SLEEP)
        if (i + 1) % 50 == 0:
            save_progress(translated_chunks, TRANSLATED_CACHE)
    save_progress(translated_chunks, TRANSLATED_CACHE)
    print(f"💾 Guardadas {len(translated_chunks)} traducciones en {TRANSLATED_CACHE}")

print(f"\n🔍 Ejemplo — chunk #50 traducido al español:\n")
print(translated_chunks[50]["text"])

🌍 Traduciendo EN→ES:   0%|          | 0/1574 [00:00<?, ?it/s]

💾 Guardadas 1574 traducciones en translated_chunks.pkl

🔍 Ejemplo — chunk #50 traducido al español:

.
En los dos capítulos siguientes, analizamos las formas en que se pueden organizar los datos dentro de un 
sistema informático. En el Capítulo 8 (Abstracciones de datos), presentamos técnicas 
tradicionalmente utilizado para organizar datos en la memoria principal de una computadora y 
luego rastrear la evolución de la abstracción de datos a partir del concepto de primitivas 
M00_BROO3427_13_GE_C00.indd 26 17/10/18 4:37 PM 270.4 Los temas generales de la informática
a las técnicas actuales orientadas a objetos. En el Capítulo 9 (Sistemas de bases de datos), 
Considere los métodos utilizados tradicionalmente para organizar datos en la masa de una computadora. 
almacenar e investigar cómo los sistemas de bases de datos extremadamente grandes y complejos 
están implementados.
En el Capítulo 10 (Gráficos por computadora), exploramos el tema de los gráficos y 
animación, campo que se ocupa 

## 🧮 Paso 7 — Embeddings con Gemini

Un **embedding** es un vector numérico que captura el significado semántico de un texto. Textos parecidos (aunque usen palabras distintas) tendrán vectores cercanos en el espacio vectorial.

Usamos el modelo **`gemini-embedding-001`** con **768 dimensiones** (buen balance entre calidad y tamaño de almacenamiento).

**Tipos de tarea (`task_type`):**
- `RETRIEVAL_DOCUMENT` → para los chunks que vamos a indexar
- `RETRIEVAL_QUERY` → para la pregunta del usuario

Esta distinción mejora la precisión: el modelo optimiza el embedding según si el texto es un "documento" o una "consulta".

Como ChromaDB no trae una `EmbeddingFunction` nativa para el SDK nuevo de Gemini, **creamos una clase custom** compatible con su interfaz.

In [9]:
import time
from google import genai
from google.genai import types
from chromadb import EmbeddingFunction, Documents, Embeddings


# Cliente único de Gemini, reutilizable
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

# Modelos
EMBED_MODEL = "gemini-embedding-001"
EMBED_DIM = 768  # 768, 1536 o 3072 (recomendado 768 por velocidad/costo)
CHAT_MODEL = "gemini-2.5-flash"

# Rate limit del free tier: 60 req/min → 1.1s deja margen
RATE_LIMIT_SLEEP = 1.1


class GeminiEmbeddingFunction(EmbeddingFunction):
    """
    Embedding function compatible con ChromaDB usando google-genai (SDK nuevo).
    Hace throttling automático para respetar el límite de 60 req/min.
    """
    def __init__(self, task_type: str = "RETRIEVAL_DOCUMENT"):
        self.task_type = task_type

    def __call__(self, input: Documents) -> Embeddings:
        embeddings = []
        for text in input:
            for attempt in range(3):
                try:
                    result = gemini_client.models.embed_content(
                        model=EMBED_MODEL,
                        contents=text,
                        config=types.EmbedContentConfig(
                            task_type=self.task_type,
                            output_dimensionality=EMBED_DIM,
                        ),
                    )
                    embeddings.append(result.embeddings[0].values)
                    time.sleep(RATE_LIMIT_SLEEP)
                    break
                except Exception as e:
                    print(f"⚠️  Reintentando ({attempt+1}/3): {str(e)[:80]}")
                    time.sleep(5)
            else:
                raise RuntimeError("❌ Embedding falló tras 3 intentos")
        return embeddings

    def name(self) -> str:
        return "gemini-embedding-001"


# Instancias para indexar (DOCUMENT) y consultar (QUERY)
embed_fn_doc = GeminiEmbeddingFunction(task_type="RETRIEVAL_DOCUMENT")
embed_fn_query = GeminiEmbeddingFunction(task_type="RETRIEVAL_QUERY")

print("✅ Embedding functions configuradas con Gemini.")
print(f"   Modelo: {EMBED_MODEL} | Dimensiones: {EMBED_DIM}")
print(f"   Rate limit: 1 request cada {RATE_LIMIT_SLEEP}s (~{int(60/RATE_LIMIT_SLEEP)}/min)")

✅ Embedding functions configuradas con Gemini.
   Modelo: gemini-embedding-001 | Dimensiones: 768
   Rate limit: 1 request cada 1.1s (~54/min)


## 🗃️ Paso 8 — ChromaDB persistente

Creamos una base de datos vectorial **persistente** (los datos se guardan en disco en `./chroma_db`). Si el kernel del notebook se reinicia, los embeddings ya calculados siguen ahí — **no hay que volver a generarlos**.

Usamos `hnsw:space=cosine` porque es la métrica más adecuada para embeddings normalizados.

In [10]:
import chromadb

CHROMA_DIR = "./chroma_db"
COLLECTION_NAME = "computer_science_book_es"

chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)

# get_or_create: si ya existe la colección, la reusa (no hay que volver a indexar)
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embed_fn_doc,
    metadata={"hnsw:space": "cosine"},
)

print(f"📦 Colección lista: '{COLLECTION_NAME}'")
print(f"   Documentos ya indexados: {collection.count()}")
print(f"   Carpeta de persistencia: {CHROMA_DIR}")

📦 Colección lista: 'computer_science_book_es'
   Documentos ya indexados: 0
   Carpeta de persistencia: ./chroma_db


## 📥 Paso 9 — Indexar los chunks traducidos en ChromaDB

Si la colección ya tiene documentos (porque corriste este paso antes), lo saltamos. Si está vacía, **indexamos los 1500 chunks** — esto invoca a Gemini Embeddings con throttling automático.

> ⏱️ **Tiempo estimado:** ~25-30 minutos para 1500 chunks (60 req/min). El progreso se guarda en disco a medida que avanza.

In [14]:
from tqdm.auto import tqdm

if collection.count() >= len(translated_chunks):
    print(f"♻️  La colección ya tiene {collection.count()} chunks indexados. Saltando indexación.")
else:
    BATCH_SIZE = 10
    start_from = collection.count()
    pending = translated_chunks[start_from:]

    print(f"📥 Indexando {len(pending)} chunks (empezando desde el #{start_from})...")
    print(f"   Esto tomará ~{len(pending) * RATE_LIMIT_SLEEP / 60:.1f} minutos.")

    for i in tqdm(range(0, len(pending), BATCH_SIZE), desc="📥 Indexando"):
        batch = pending[i:i + BATCH_SIZE]
        global_idx = start_from + i
        collection.add(
            documents=[c["text"] for c in batch],
            metadatas=[c["metadata"] for c in batch],
            ids=[f"chunk_{global_idx + j}" for j in range(len(batch))],
        )

    print(f"\n✅ Indexación completa: {collection.count()} chunks en ChromaDB.")

📥 Indexando 584 chunks (empezando desde el #990)...
   Esto tomará ~10.7 minutos.


📥 Indexando:   0%|          | 0/59 [00:00<?, ?it/s]

⚠️  Reintentando (1/3): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your cu
⚠️  Reintentando (2/3): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your cu
⚠️  Reintentando (3/3): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your cu


RuntimeError: ❌ Embedding falló tras 3 intentos in add.

## 🔍 Paso 10 — Búsqueda semántica + Generación de respuesta

Definimos dos funciones núcleo del sistema RAG:

1. **`semantic_search(question, k)`** — embebe la pregunta con `task_type=RETRIEVAL_QUERY` y devuelve los `k` chunks más similares.
2. **`answer_with_context(question)`** — toma esos chunks, los pasa como contexto al LLM `gemini-2.5-flash` y genera una respuesta en español que **se basa solo en el libro** (esto reduce alucinaciones).

In [15]:
def semantic_search(question: str, k: int = 5) -> list[dict]:
    """Busca los k chunks más relevantes para la pregunta."""
    # Embedding manual con task_type=QUERY para mejor precisión
    query_embedding = embed_fn_query([question])[0]
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k,
    )
    return [
        {"text": doc, "metadata": meta, "distance": dist}
        for doc, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        )
    ]


SYSTEM_PROMPT = """Eres un asistente experto en Ciencias de la Computación. Respondes
preguntas sobre el libro 'Computer Science: An Overview' (13ª edición) basándote
EXCLUSIVAMENTE en los fragmentos de contexto que se te proporcionan.

REGLAS:
- Responde en español, de forma clara y didáctica.
- Si la respuesta NO está en los fragmentos, di: "No encontré esa información en el libro."
- No inventes datos. No alucines.
- Cita brevemente de qué parte del libro proviene la información cuando sea útil."""


def answer_with_context(question: str, k: int = 5) -> dict:
    """Pipeline RAG completo: búsqueda + generación."""
    # 1. Recuperar contexto
    retrieved = semantic_search(question, k=k)
    context = "\n\n---\n\n".join(
        f"[Fragmento {i+1}] {r['text']}" for i, r in enumerate(retrieved)
    )

    # 2. Generar respuesta con Gemini Flash
    prompt = f"""{SYSTEM_PROMPT}

CONTEXTO DEL LIBRO:
{context}

PREGUNTA DEL USUARIO:
{question}

RESPUESTA:"""

    response = gemini_client.models.generate_content(
        model=CHAT_MODEL,
        contents=prompt,
    )

    return {
        "answer": response.text,
        "sources": retrieved,
    }


# Probemos rápido sin UI
print("🧪 Prueba rápida del pipeline RAG:\n")
test = answer_with_context("¿Qué es un sistema operativo?", k=3)
print("📝 RESPUESTA:\n", test["answer"])
print("\n📚 FUENTES (top 3):")
for i, src in enumerate(test["sources"], 1):
    print(f"   {i}. (distancia={src['distance']:.4f}) {src['text'][:120]}...")

🧪 Prueba rápida del pipeline RAG:

⚠️  Reintentando (1/3): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your cu
⚠️  Reintentando (2/3): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your cu
⚠️  Reintentando (3/3): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your cu


RuntimeError: ❌ Embedding falló tras 3 intentos

## 🎨 Paso 11 — Casilla estética con `ipywidgets`

Aquí está la **interfaz interactiva** que pediste: un cuadro de texto, un botón, y un panel de respuesta con buen formato. Todo dentro del notebook, sin necesidad de salir.

> 💡 En VS Code, los widgets se renderizan directamente en el panel del notebook. Si no los ves, abre la paleta de comandos (`Ctrl+Shift+P`) y busca `Jupyter: Restart Kernel`, luego vuelve a ejecutar.

In [16]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# === Componentes visuales ===
title = widgets.HTML("""
<div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            padding: 20px; border-radius: 12px; color: white;
            font-family: -apple-system, sans-serif; margin-bottom: 12px;'>
    <h2 style='margin:0;'>📚 Pregunta al libro de Ciencias de la Computación</h2>
    <p style='margin:6px 0 0 0; opacity:0.9;'>
        Sistema RAG — Computer Science: An Overview (13th ed.)
    </p>
</div>
""")

question_box = widgets.Textarea(
    placeholder="Escribe tu pregunta aquí... ej: ¿Qué es la complejidad algorítmica?",
    layout=widgets.Layout(width="100%", height="80px"),
    description="❓",
)

k_slider = widgets.IntSlider(
    value=5, min=1, max=10, step=1,
    description="Fragmentos a recuperar:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="50%"),
)

ask_button = widgets.Button(
    description="🚀 Preguntar",
    button_style="primary",
    layout=widgets.Layout(width="180px", height="40px"),
)

clear_button = widgets.Button(
    description="🧹 Limpiar",
    button_style="",
    layout=widgets.Layout(width="120px", height="40px"),
)

output_area = widgets.Output()


def render_answer(question: str, result: dict):
    """Renderiza la respuesta con HTML bonito."""
    sources_html = ""
    for i, src in enumerate(result["sources"], 1):
        sources_html += f"""
        <details style='margin: 8px 0; padding: 10px; background: #f7f8fa;
                        border-left: 3px solid #667eea; border-radius: 6px;'>
            <summary style='cursor: pointer; font-weight: 600; color: #555;'>
                📄 Fragmento {i} — distancia: {src['distance']:.4f}
            </summary>
            <p style='margin-top: 8px; color: #333; font-size: 0.9em; line-height: 1.5;'>
                {src['text']}
            </p>
        </details>
        """

    html = f"""
    <div style='font-family: -apple-system, sans-serif; max-width: 900px;'>
        <div style='background:#fff;padding:18px;border-radius:10px;
                    border:1px solid #e0e0e0;margin-bottom:14px;
                    box-shadow:0 2px 8px rgba(0,0,0,0.04);'>
            <div style='color:#667eea;font-weight:600;font-size:0.85em;
                        text-transform:uppercase;letter-spacing:0.5px;'>
                Tu pregunta
            </div>
            <div style='font-size:1.1em;color:#222;margin-top:6px;'>{question}</div>
        </div>

        <div style='background:#fff;padding:20px;border-radius:10px;
                    border:1px solid #e0e0e0;margin-bottom:14px;
                    box-shadow:0 2px 8px rgba(0,0,0,0.04);'>
            <div style='color:#10b981;font-weight:600;font-size:0.85em;
                        text-transform:uppercase;letter-spacing:0.5px;'>
                💡 Respuesta del libro
            </div>
            <div style='font-size:1.05em;color:#222;margin-top:10px;
                        line-height:1.7;white-space:pre-wrap;'>{result["answer"]}</div>
        </div>

        <div style='background:#fff;padding:16px;border-radius:10px;
                    border:1px solid #e0e0e0;'>
            <div style='color:#888;font-weight:600;font-size:0.85em;
                        text-transform:uppercase;letter-spacing:0.5px;
                        margin-bottom:8px;'>
                🔍 Fuentes consultadas
            </div>
            {sources_html}
        </div>
    </div>
    """
    display(HTML(html))


def on_ask_clicked(_):
    q = question_box.value.strip()
    if not q:
        with output_area:
            clear_output()
            print("⚠️  Escribe una pregunta primero.")
        return
    with output_area:
        clear_output()
        print("⏳ Buscando en el libro y generando respuesta...")
        result = answer_with_context(q, k=k_slider.value)
        clear_output()
        render_answer(q, result)


def on_clear_clicked(_):
    question_box.value = ""
    with output_area:
        clear_output()


ask_button.on_click(on_ask_clicked)
clear_button.on_click(on_clear_clicked)

# === Layout final ===
ui = widgets.VBox([
    title,
    question_box,
    widgets.HBox([k_slider]),
    widgets.HBox([ask_button, clear_button]),
    output_area,
])

display(ui)

## ✅ ¡Listo!

Has construido un sistema RAG completo end-to-end:

| Etapa | Tecnología |
|---|---|
| Extracción | `pypdf` |
| Tokenización | `tiktoken` |
| Chunking | `langchain RecursiveCharacterTextSplitter` |
| Traducción | `deep-translator` (Google Translate) |
| Embeddings | `gemini-embedding-001` (768 dim) |
| Vector DB | `ChromaDB` persistente |
| LLM | `gemini-2.5-flash` |
| UI | `ipywidgets` |

### 💡 Ideas para extender el proyecto

- **Streaming de respuestas** con `gemini_client.models.generate_content_stream(...)`
- **Filtros por capítulo** añadiendo `chapter` al metadata
- **Re-ranking** de los chunks recuperados con un cross-encoder
- **Multi-query**: generar 3 reformulaciones de la pregunta y combinar resultados
- **Evaluación**: medir precisión con un set de Q&A de referencia

---

📦 **Repo:** *(pega aquí la URL de tu repo de GitHub cuando lo publiques)*